In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Load dataset
df = pd.read_csv(r"C:\Users\SCA_11\Downloads\products.csv")

# Select columns
df = df[['asin', 'title', 'about_item', 'rating_stars', 'price_value']]

# Fill missing values
df['title'] = df['title'].fillna('')
df['about_item'] = df['about_item'].fillna('')
df['rating_stars'] = df['rating_stars'].fillna('0')
df['price_value'] = df['price_value'].fillna('0')

# Reset index
df = df.reset_index(drop=True)

# Combine title and about_item
df['content'] = df['title'] + " " + df['about_item']

print("Dataset Preview:")
print(df.head())

print("\nAvailable Product Titles:")
print(df['title'].head(10))

# -------------------------------
# CONTENT BASED RECOMMENDATION
# -------------------------------

tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
text_matrix = tfidf.fit_transform(df['content'])

similarity = cosine_similarity(text_matrix)

def recommend(product_name, top_n=5):
    matched_products = df[
        df['title'].str.contains(product_name, case=False, na=False)
    ]

    if matched_products.empty:
        print("\nNo product found with this name.")
        print("\nTry one of these product titles:")
        print(df['title'].head(10))
        return

    idx = matched_products.index[0]

    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = scores[1:top_n + 1]

    print("\nSelected Product:")
    print(df.loc[idx, 'title'])

    print("\nRecommended Products:")
    for i in scores:
        print(df.iloc[i[0]]['title'])


# Auto example using first product word
first_word = df['title'].iloc[0].split()[0]
recommend(first_word)


# -------------------------------
# MATRIX FACTORIZATION USING SVD
# -------------------------------

svd_components = min(5, text_matrix.shape[1] - 1)

svd = TruncatedSVD(n_components=svd_components, random_state=42)
svd_matrix = svd.fit_transform(text_matrix)

print("\nSVD Matrix Factorization Output:")
print(svd_matrix[:10])

print("\nShape of SVD Matrix:")
print(svd_matrix.shape)

Dataset Preview:
         asin                                              title  \
0  B0B59BJG6Y  4/5 Pack Mens Polo Shirts Short Sleeve Quick D...   
1  B0DLGB4RYH  COOFANDY Men's Polo Shirts Short Sleeve Moistu...   
2  B0DRXF62JH  ZITY 3 Pack Men Polo Shirts Short Sleeve with ...   
3  B0DK5FZ325  Rouen Mens Golf Shirt Moisture Wicking Dry Fit...   
4  B0BGXTC1FR  V VALANCH Mens Polo Shirts Short Sleeve Moistu...   

                                          about_item        rating_stars  \
0  Premium Comfort: Crafted from a high-quality c...  4.6 out of 5 stars   
1  Material: Men's polo shirt is made of soft pol...  4.4 out of 5 stars   
2  PERFORMANCE:These men polo shirts are soft,lig...  4.6 out of 5 stars   
3  【Material】: These golf shirts for men are made...  4.8 out of 5 stars   
4  MOISTURE WICKING: The fabric of the summer gol...  4.4 out of 5 stars   

  price_value                                            content  
0     39.9926  4/5 Pack Mens Polo Shirts Short Sle